In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torchaudio
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.DataStructs.cDataStructs import ConvertToNumpyArray
from encodec import EncodecModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Inference Mode: Operating safely on {device}")

In [ ]:
# --- 1. RDKit Morgan System & PyTorch Transformers ---
def morgan_fp_from_smiles(smiles, n_bits=600, radius=2):
    arr = np.zeros(n_bits, dtype=np.float32)
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        ConvertToNumpyArray(fp, arr)
    return arr

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=20000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class Polymer2Audio(nn.Module):
    def __init__(self, fp_dim=600, vocab_size=2000, d_model=512, nhead=8, num_layers=4, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.fp_proj = nn.Linear(fp_dim, d_model)
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def generate_square_subsequent_mask(self, sz, device):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask.to(device)

    def forward(self, polymer_fp, tgt_tokens, tgt_padding_mask=None):
        device = tgt_tokens.device
        memory = self.fp_proj(polymer_fp).unsqueeze(1)
        tgt_emb = self.token_embedding(tgt_tokens) * math.sqrt(self.d_model)
        tgt_emb = self.pos_encoder(tgt_emb)
        tgt_mask = self.generate_square_subsequent_mask(tgt_tokens.size(1), device)
        
        output = self.transformer_decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )
        return self.fc_out(output)

# Instantiating the trained Model Space!
model = Polymer2Audio(fp_dim=600, vocab_size=2000, d_model=512)
weights_path = "/home/krishna/polymer_soni1_weights.pt"

if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print("Successfully mapped previously trained Transformer Architecture Weights!")
else:
    print(f"CRITICAL WARNING: '{weights_path}' does not exist! Please ensure you have run the training notebook and generated this weights file first!")

model.to(device)
model.eval()

# Preparing EnCodec Engine Target Backend
print("Mounting Meta EnCodec 24kHz Backend...")
encodec_model = EncodecModel.encodec_model_24khz()
encodec_model.set_target_bandwidth(6.0)
encodec_model.to(device)
encodec_model.eval()
print("Success!")

In [ ]:
# --- 2. Inference Function (PSMILES -> WAV) ---

def generate_audio_from_smiles(smiles, output_wav_name="generated_polymer.wav", max_length=500):
    """Translates PSMILES natively into mathematical audio representations!"""
    
    # 1. Feature Phase
    fp_array = morgan_fp_from_smiles(smiles)
    fp_tensor = torch.tensor(fp_array, dtype=torch.float32).unsqueeze(0).to(device) # (1, 600)
    
    # 2. Composition Phase (Autoregressively generating notes)
    # Token Range 1-1024 are valid indices. We provide '1' as our starting Sequence seed.
    generated = [1] 
    
    print(f"🧠 Composing Autoregressive Tokens for '{smiles}'...")
    with torch.no_grad():
        for i in range(max_length):
            tgt_tensor = torch.tensor(generated, dtype=torch.long).unsqueeze(0).to(device) # (1, seq_len)
            logits = model(fp_tensor, tgt_tensor)
            
            # Extract probability distributions on the absolute LAST token guess
            next_token_logits = logits[0, -1, :] 
            next_token = torch.argmax(next_token_logits).item()
            
            if next_token == 0: 
                # Reached padding limit marker (0)! The model decides sequence logically concluded!
                print(f"Model declared song over naturally at index {len(generated)}.")
                break
                
            generated.append(next_token)
            
    print(f"Acoustic Token Map finished: {len(generated)} contiguous tokens!")
    
    # 3. Audio Decoding Phase
    # Remove the starting seed token '1' we used to kick off the engine
    pure_tokens = generated[1:]
    
    # Math Offset Safety: model outputs absolute indices 1-1024, shift them strictly back to EnCodec format (0-1023)
    # CRITICAL FIX: Math bounds capping limits explicitly enforced here
    converted_tokens = [max(0, min(1023, token - 1)) for token in pure_tokens]
    
    # Mathematical Array Unflattening!
    CODEBOOKS = 8 # EnCodec internally uses 8 parallel codebooks when compressed at 6.0 bandwidth rating!
    
    if len(converted_tokens) % CODEBOOKS != 0:
        # Snap tensor array length to fit the modulus safely
        bound = (len(converted_tokens) // CODEBOOKS) * CODEBOOKS
        converted_tokens = converted_tokens[:bound]
        
    num_steps = len(converted_tokens) // CODEBOOKS
    
    # Shape Transformation mapping sequentially backwards logic: (num_steps, 8) -> (8, num_steps) -> (1, 8, num_steps)
    sequence_tensor = torch.tensor(converted_tokens, dtype=torch.int64).to(device)
    code_tensor = sequence_tensor.view(num_steps, CODEBOOKS).transpose(0, 1).unsqueeze(0).contiguous()
        
    encoded_frames = [(code_tensor, None)]
    
    print("🎵 Unpacking compressed acoustic tokens -> Real .wav audio tensor")
    with torch.no_grad():
        reconstructed_audio = encodec_model.decode(encoded_frames).detach().cpu()
        
    # Torchaudio natively processes sequence extractions dynamically back to the SSD safely
    torchaudio.save(output_wav_name, reconstructed_audio.squeeze(0), sample_rate=encodec_model.sample_rate)
    print(f"✨ Phase 3 Engine Completed! Song successfully deployed: '{output_wav_name}'")


# -------------------------
# Test Your Prediction API!
# -------------------------
test_psmiles = "[*]CC([*])C"
print(f"Starting pipeline prediction execution on: {test_psmiles}")
generate_audio_from_smiles(test_psmiles, output_wav_name="predict_CC_C.wav", max_length=500)
